# LeetCode 1257 — Smallest Common Region bằng Python

Link đề: https://leetcode.com/problems/smallest-common-region/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, String, Tree, DFS, BFS, Lowest Common Ancestor · 🔒 **Premium**

Skeleton LeetCode (theo lời giải Python trên doocs):

```python
class Solution:
    def findSmallestRegion(
        self, regions: list[list[str]], region1: str, region2: str
    ) -> str:
        ...
```

> **Ghi chú Premium:** đề bài trong notebook được viết lại theo hiểu biết về bài (tham khảo https://leetcode.doocs.org/en/lc/1257/). Nếu có tài khoản Premium, hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

1. Nhận ra rằng quan hệ "vùng chứa vùng" tạo thành một **cây có gốc**, và "vùng nhỏ nhất chứa cả hai" chính là **tổ tiên chung thấp nhất (LCA)**.
2. Biết dựng **bảng cha** `parent[child] = cha` bằng `dict` từ dữ liệu dạng danh sách.
3. Cài đặt 4 cách tìm LCA trên cây cha: thử từng tổ tiên (brute force), **tập tổ tiên**, **cân bằng độ sâu**, **hai con trỏ đổi đường**, và DFS từ gốc.
4. Hiểu khi nào cần kỹ thuật mạnh hơn (**binary lifting**) — khi có rất nhiều truy vấn.
5. Tránh lỗi: quên "một vùng chứa chính nó", không đưa gốc vào tập tổ tiên, dựng bảng cha sai chiều.

## 2. Đề bài & ví dụ

Cho một số danh sách vùng `regions`; trong mỗi danh sách, **vùng đầu tiên chứa trực tiếp** tất cả các vùng còn lại của danh sách đó.

- Nếu `x` chứa trực tiếp `y` và `y` chứa trực tiếp `z` thì `x` chứa `z` (gián tiếp). Quan hệ chứa có tính bắc cầu.
- Theo định nghĩa, **mỗi vùng chứa chính nó**.
- Vùng chứa vùng khác thì lớn hơn hoặc bằng về kích thước.

Cho hai vùng `region1`, `region2`, trả về **vùng nhỏ nhất chứa cả hai**. Đề đảm bảo đáp án tồn tại.

**Ví dụ 1:** với `regions` như dưới, `region1 = "Quebec"`, `region2 = "New York"` → `"North America"`.

**Ví dụ 2:** cùng `regions`, `region1 = "Canada"`, `region2 = "South America"` → `"Earth"`.

**Ràng buộc:**
- `2 <= regions.length <= 10^4`, `2 <= regions[i].length <= 20`
- Tên vùng dài 1..20, chỉ gồm chữ cái tiếng Anh; `region1 != region2`.
- Tồn tại một vùng chứa mọi vùng khác (gốc); **mỗi vùng bị chứa trực tiếp bởi tối đa một vùng**.

In [1]:
import random
import sys
import time
from collections import defaultdict, deque

REGIONS = [["Earth", "North America", "South America"],
           ["North America", "United States", "Canada"],
           ["United States", "New York", "Boston"],
           ["Canada", "Ontario", "Quebec"],
           ["South America", "Brazil"]]
examples = [(REGIONS, "Quebec", "New York", "North America"),
            (REGIONS, "Canada", "South America", "Earth")]
for regs, a, b, want in examples:
    print(f"{a!r:15} + {b!r:15} -> {want!r}")

'Quebec'        + 'New York'      -> 'North America'
'Canada'        + 'South America' -> 'Earth'


## 3. Kiến thức nền

### 3.1 Cây có gốc và bảng cha

Mỗi vùng (trừ gốc) có **đúng một** vùng chứa trực tiếp nó ⇒ đó là **cha**. Lưu bằng `dict`: `parent["Quebec"] = "Canada"`. Gốc là vùng **không có** trong `parent`.

Từ một danh sách `r = [x, y1, y2, ...]`: `r[0]` là cha, `r[1:]` là các con.

In [2]:
parent = {}
for r in REGIONS:
    for child in r[1:]:
        parent[child] = r[0]
print(parent)
roots = {r[0] for r in REGIONS} - set(parent)
print("Gốc:", roots)

{'North America': 'Earth', 'South America': 'Earth', 'United States': 'North America', 'Canada': 'North America', 'New York': 'United States', 'Boston': 'United States', 'Ontario': 'Canada', 'Quebec': 'Canada', 'Brazil': 'South America'}
Gốc: {'Earth'}


### 3.2 Đi ngược lên gốc — đường tổ tiên

Từ một vùng, lặp `x = parent[x]` cho đến khi `x` không còn cha. Các vùng đi qua (gồm chính nó) là **các vùng chứa nó**, theo thứ tự từ nhỏ đến lớn.

In [3]:
def path_to_root(x, parent):
    path = [x]
    while x in parent:
        x = parent[x]
        path.append(x)
    return path

print(path_to_root("Quebec", parent))
print(path_to_root("New York", parent))
print(path_to_root("Earth", parent))

['Quebec', 'Canada', 'North America', 'Earth']
['New York', 'United States', 'North America', 'Earth']
['Earth']


### 3.3 `set` — kiểm tra thuộc trong O(1) trung bình

`x in s` với `s` là `set` mất trung bình `O(1)`, còn với `list` mất `O(độ dài)`. Đây là chìa khoá của lời giải chuẩn.

In [4]:
anc = set(path_to_root("Quebec", parent))
print(anc)
print("'Canada' in anc:", "Canada" in anc, "| 'Boston' in anc:", "Boston" in anc)

{'Earth', 'North America', 'Canada', 'Quebec'}
'Canada' in anc: True | 'Boston' in anc: False


### 3.4 Danh sách con & DFS đệ quy

Để đi **từ gốc xuống**, ta cần `children[x] = [các con]`. `defaultdict(list)` giúp không phải kiểm tra khoá tồn tại.

In [5]:
children = defaultdict(list)
for r in REGIONS:
    children[r[0]].extend(r[1:])

def print_tree(x, depth=0):
    print("    " * depth + x)
    for c in children[x]:
        print_tree(c, depth + 1)

print_tree("Earth")

Earth
    North America
        United States
            New York
            Boston
        Canada
            Ontario
            Quebec
    South America
        Brazil


### 3.5 Độ sâu (depth)

Độ sâu của vùng = số bước đi lên tới gốc. Gốc có độ sâu 0. Hai vùng muốn "gặp nhau" khi đi lên thì phải ở cùng độ sâu trước.

In [6]:
def depth(x, parent):
    d = 0
    while x in parent:
        x = parent[x]
        d += 1
    return d

for x in ["Earth", "Canada", "Quebec", "Brazil"]:
    print(f"{x:15} depth = {depth(x, parent)}")

Earth           depth = 0
Canada          depth = 2
Quebec          depth = 3
Brazil          depth = 2


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1** (`Quebec`, `New York`):

- Các vùng chứa `Quebec`: `Quebec → Canada → North America → Earth`.
- Các vùng chứa `New York`: `New York → United States → North America → Earth`.
- Vùng chứa **cả hai**: `{North America, Earth}`. Vùng **nhỏ nhất** là cái gần lá nhất ⇒ `North America`.

**Ví dụ 2** (`Canada`, `South America`): `Canada → North America → Earth`; `South America → Earth` ⇒ chung `{Earth}` ⇒ `Earth`.

Trường hợp đặc biệt: nếu `region1` chứa `region2` (vd. `Canada`, `Quebec`) thì đáp án là chính `Canada` — vì **vùng chứa chính nó**.

In [7]:
def common_ancestors(a, b, parent):
    pa, pb = path_to_root(a, parent), path_to_root(b, parent)
    return [x for x in pa if x in pb]          # theo thứ tự từ nhỏ đến lớn

for a, b in [("Quebec", "New York"), ("Canada", "South America"), ("Canada", "Quebec"), ("Boston", "Brazil")]:
    common = common_ancestors(a, b, parent)
    print(f"{a:10} {b:14} chung = {common} -> nhỏ nhất: {common[0]}")

Quebec     New York       chung = ['North America', 'Earth'] -> nhỏ nhất: North America
Canada     South America  chung = ['Earth'] -> nhỏ nhất: Earth
Canada     Quebec         chung = ['Canada', 'North America', 'Earth'] -> nhỏ nhất: Canada
Boston     Brazil         chung = ['Earth'] -> nhỏ nhất: Earth


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** Với mỗi vùng `x` trên đường từ `region1` lên gốc (theo thứ tự từ dưới lên), kiểm tra `x` có chứa `region2` không bằng cách **đi từ `region2` lên gốc** và xem có gặp `x`. Vùng đầu tiên thoả là đáp án.

Không dùng `set`, mỗi lần kiểm tra lại đi một đường dài.

In [8]:
def build_parent(regions):
    parent = {}
    for r in regions:
        for child in r[1:]:
            parent[child] = r[0]
    return parent

def contains(x, y, parent):
    """x có chứa y không? (đi từ y lên gốc tìm x)"""
    while True:
        if y == x:
            return True
        if y not in parent:
            return False
        y = parent[y]

def smallest_brute(regions, region1, region2):
    parent = build_parent(regions)
    x = region1
    while True:
        if contains(x, region2, parent):
            return x
        x = parent[x]          # đảm bảo có đáp án nên không đi quá gốc

for regs, a, b, want in examples:
    print(smallest_brute(regs, a, b), "| mong đợi", want)

North America | mong đợi North America
Earth | mong đợi Earth


**Độ phức tạp:** gọi `N` = tổng số vùng (≤ 20·10⁴), `h` = chiều cao cây. Dựng bảng cha `O(N)`; vòng ngoài ≤ `h` lần, mỗi lần `contains` tốn ≤ `h` ⇒ **`O(N + h²)`** thời gian, `O(N)` bộ nhớ. Cây suy biến thành "dây chuyền" có `h ≈ N` ⇒ `O(N²)`.

**Nút thắt:** lặp đi lặp lại đường từ `region2` lên gốc. Chỉ cần đi **một lần** và nhớ vào `set`.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** vùng `x` chứa `y` ⇔ `x` nằm trên đường từ `y` lên gốc. Do đó:

> tập vùng chứa cả hai = (tổ tiên của `region1`) ∩ (tổ tiên của `region2`),

và tập này là một **đoạn đầu chung** của hai đường tính từ gốc xuống. Vùng nhỏ nhất là phần tử **thấp nhất** (xa gốc nhất) của đoạn đó.

**Quy tắc "nếu… thì…":**
- Nếu đã lưu mọi tổ tiên của `region1` vào `set S` thì đi từ `region2` lên, **vùng đầu tiên thuộc `S`** là đáp án.
- Nếu hai vùng ở **cùng độ sâu** thì cứ đi lên đồng thời, lần đầu chúng trùng nhau là đáp án.
- Nếu có **nhiều truy vấn** trên cùng cây thì tiền xử lý (binary lifting / Euler tour) để mỗi truy vấn `O(log N)`.

**Invariant (cách tập tổ tiên):** khi đi từ `region2` lên, mọi vùng đã đi qua mà chưa thuộc `S` đều **không** chứa `region1`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tập tổ tiên (lời giải doocs)

1. Dựng `g[child] = parent`.
2. Đi từ `region1` lên, thêm mọi vùng vào `s`.
3. Đi từ `region2` lên đến khi gặp vùng thuộc `s` (hoặc tới gốc).

Lưu ý cách doocs viết: `while x in g: s.add(x)` — vòng lặp dừng **trước** khi thêm gốc. Không sao, vì ở bước 3 điều kiện `while x in g and x not in s` cũng dừng khi tới gốc ⇒ gốc là đáp án mặc định.

In [9]:
def smallest_ancestor_set(regions, region1, region2):
    g = build_parent(regions)
    s = set()
    x = region1
    while x in g:
        s.add(x)
        x = g[x]
    x = region2
    while x in g and x not in s:
        x = g[x]
    return x

for regs, a, b, want in examples:
    print(smallest_ancestor_set(regs, a, b), "| mong đợi", want)

North America | mong đợi North America
Earth | mong đợi Earth


Bản nộp LeetCode (đúng tên lớp/hàm/tham số như doocs):

In [10]:
class SolutionSet:
    def findSmallestRegion(
        self, regions: list[list[str]], region1: str, region2: str
    ) -> str:
        g = {}
        for r in regions:
            x = r[0]
            for y in r[1:]:
                g[y] = x
        s = set()
        x = region1
        while x in g:
            s.add(x)
            x = g[x]
        x = region2
        while x in g and x not in s:
            x = g[x]
        return x

print([SolutionSet().findSmallestRegion(*e[:3]) for e in examples])

['North America', 'Earth']


**Độ phức tạp Cách 2:** `O(N)` dựng bảng + `O(h)` hai lần đi lên ⇒ **`O(N)` thời gian**, **`O(N)` bộ nhớ**.

### 7.2 Cách 3 — Cân bằng độ sâu rồi đi lên đồng thời

Không dùng tập tổ tiên:
1. Tính `da = depth(region1)`, `db = depth(region2)`.
2. Đưa vùng sâu hơn đi lên `|da − db|` bước.
3. Khi cùng độ sâu: trong khi `a != b`, cho cả hai lên cha. Điểm gặp là LCA.

Đây là kỹ thuật LCA "ngây thơ" kinh điển, và là nền tảng của binary lifting.

In [11]:
def smallest_depth(regions, region1, region2):
    parent = build_parent(regions)
    def depth(x):
        d = 0
        while x in parent:
            x = parent[x]; d += 1
        return d
    a, b = region1, region2
    da, db = depth(a), depth(b)
    while da > db:
        a = parent[a]; da -= 1
    while db > da:
        b = parent[b]; db -= 1
    while a != b:
        a, b = parent[a], parent[b]
    return a

for regs, a, b, want in examples:
    print(smallest_depth(regs, a, b), "| mong đợi", want)

North America | mong đợi North America
Earth | mong đợi Earth


**Độ phức tạp Cách 3:** `O(N + h)` thời gian; bộ nhớ phụ ngoài bảng cha chỉ **`O(1)`**.

### 7.3 Cách 4 — Hai con trỏ "đổi đường" (giống LeetCode 160)

Hai đường từ `region1` và `region2` lên gốc giống hai danh sách liên kết **giao nhau** tại LCA. Cho con trỏ `a` đi từ `region1`, `b` đi từ `region2`; khi một con trỏ **đi quá gốc** thì nhảy sang điểm xuất phát của bên kia. Sau tối đa `da + db + 2` bước chúng gặp nhau đúng tại LCA, vì cả hai đã đi cùng tổng quãng đường `da + 1 + db + 1`... Ta dùng `None` để đánh dấu "quá gốc".

In [12]:
def smallest_two_pointers(regions, region1, region2):
    parent = build_parent(regions)
    a, b = region1, region2
    while a != b:
        a = parent.get(a, None) if a is not None else region2
        b = parent.get(b, None) if b is not None else region1
    return a

for regs, a, b, want in examples:
    print(smallest_two_pointers(regs, a, b), "| mong đợi", want)

North America | mong đợi North America
Earth | mong đợi Earth


Vì sao gặp đúng LCA? Gọi `p` = số bước từ `region1` tới LCA, `q` = từ `region2` tới LCA, `t` = từ LCA tới "quá gốc" (`None`). Con trỏ `a` đi `p + t` bước rồi nhảy sang `region2`, đi thêm `q` bước nữa là tới LCA: tổng `p + t + q`. Con trỏ `b` cũng tới LCA sau `q + t + p` bước. Hai con trỏ không thể gặp nhau sớm hơn ở một vùng khác LCA vì trước LCA hai đường rời nhau (nếu `p == q` chúng gặp ngay lần đầu).

**Độ phức tạp Cách 4:** `O(N + h)` thời gian, `O(1)` bộ nhớ phụ.

### 7.4 Cách 5 — DFS từ gốc (LCA trên cây tổng quát)

Dựng `children`, tìm gốc, rồi DFS trả về **số mục tiêu** (`region1`, `region2`) có trong cây con. Nút **đầu tiên** (sâu nhất) mà cây con chứa đủ 2 mục tiêu là đáp án. Dùng stack hậu thứ tự để tránh đệ quy sâu.

In [13]:
def smallest_dfs(regions, region1, region2):
    children = defaultdict(list)
    has_parent = set()
    for r in regions:
        children[r[0]].extend(r[1:])
        has_parent.update(r[1:])
    root = next(r[0] for r in regions if r[0] not in has_parent)
    targets = {region1, region2}
    count = {}
    stack = [(root, False)]
    while stack:
        x, done = stack.pop()
        if not done:
            stack.append((x, True))
            for c in children[x]:
                stack.append((c, False))
        else:
            cnt = (x in targets) + sum(count[c] for c in children[x])
            count[x] = cnt
            if cnt == 2:
                return x                # nút hậu thứ tự đầu tiên đủ 2 là sâu nhất
    return root

for regs, a, b, want in examples:
    print(smallest_dfs(regs, a, b), "| mong đợi", want)

North America | mong đợi North America
Earth | mong đợi Earth


Vì sao "nút hậu thứ tự đầu tiên có `cnt == 2`" là LCA? Trong hậu thứ tự, một nút chỉ được xử lý sau **toàn bộ** cây con của nó. Mọi nút có `cnt == 2` đều là tổ tiên chung (nằm trên đường LCA → gốc), và trong đó LCA được xử lý trước các tổ tiên của nó.

**Độ phức tạp Cách 5:** duyệt cả cây ⇒ `O(N)` thời gian, `O(N)` bộ nhớ. Chậm hơn hằng số so với Cách 2 nhưng là khuôn mẫu LCA tổng quát (bài 236, 1650...).

## 8. TRACE từng bước

Trace **Cách 2** với `region1 = "Quebec"`, `region2 = "New York"`:

| Pha | `x` | hành động | `s` |
|---|---|---|---|
| 1 | Quebec | thêm, lên Canada | {Quebec} |
| 1 | Canada | thêm, lên North America | {Quebec, Canada} |
| 1 | North America | thêm, lên Earth | {…, North America} |
| 1 | Earth | không có cha ⇒ dừng | (gốc không thêm) |
| 2 | New York | ∉ s, lên United States | |
| 2 | United States | ∉ s, lên North America | |
| 2 | North America | **∈ s ⇒ dừng** | đáp án |

In [14]:
def trace_ancestor_set(regions, region1, region2):
    g = build_parent(regions)
    s = set()
    x = region1
    print("Pha 1: đi từ", region1)
    while x in g:
        s.add(x)
        print(f"   thêm {x!r:16} -> lên {g[x]!r:16} s={sorted(s)}")
        x = g[x]
    print(f"   {x!r} là gốc, dừng pha 1")
    x = region2
    print("Pha 2: đi từ", region2)
    while x in g and x not in s:
        print(f"   {x!r:16} không thuộc s -> lên {g[x]!r}")
        x = g[x]
    print(f"   dừng tại {x!r} ({'thuộc s' if x in s else 'là gốc'})")
    return x

print("Kết quả:", trace_ancestor_set(REGIONS, "Quebec", "New York"))

Pha 1: đi từ Quebec
   thêm 'Quebec'         -> lên 'Canada'         s=['Quebec']
   thêm 'Canada'         -> lên 'North America'  s=['Canada', 'Quebec']
   thêm 'North America'  -> lên 'Earth'          s=['Canada', 'North America', 'Quebec']
   'Earth' là gốc, dừng pha 1
Pha 2: đi từ New York
   'New York'       không thuộc s -> lên 'United States'
   'United States'  không thuộc s -> lên 'North America'
   dừng tại 'North America' (thuộc s)
Kết quả: North America


In [15]:
def trace_depth(regions, region1, region2):
    parent = build_parent(regions)
    def depth(x):
        d = 0
        while x in parent:
            x = parent[x]; d += 1
        return d
    a, b = region1, region2
    da, db = depth(a), depth(b)
    print(f"bắt đầu: a={a!r} (sâu {da}), b={b!r} (sâu {db})")
    while da > db:
        a = parent[a]; da -= 1
        print(f"   nâng a -> {a!r} (sâu {da})")
    while db > da:
        b = parent[b]; db -= 1
        print(f"   nâng b -> {b!r} (sâu {db})")
    while a != b:
        a, b = parent[a], parent[b]
        print(f"   cùng lên: a={a!r}, b={b!r}")
    print("gặp nhau tại", repr(a))
    return a

trace_depth(REGIONS, "Boston", "Canada")

bắt đầu: a='Boston' (sâu 3), b='Canada' (sâu 2)
   nâng a -> 'United States' (sâu 2)
   cùng lên: a='North America', b='North America'
gặp nhau tại 'North America'


'North America'

## 9. Code nộp LeetCode

Đúng tên lớp, phương thức, tham số theo lời giải Python trên doocs:

In [16]:
class Solution:
    def findSmallestRegion(
        self, regions: list[list[str]], region1: str, region2: str
    ) -> str:
        g = {}
        for r in regions:
            x = r[0]
            for y in r[1:]:
                g[y] = x                     # bảng cha
        s = set()
        x = region1
        while x in g:                        # mọi vùng chứa region1 (trừ gốc)
            s.add(x)
            x = g[x]
        x = region2
        while x in g and x not in s:         # vùng đầu tiên chứa region1
            x = g[x]
        return x

for regs, a, b, want in examples:
    got = Solution().findSmallestRegion(regs, a, b)
    print(got, "OK" if got == want else "SAI")

North America OK
Earth OK


## 10. Vì sao thuật toán đúng

**Bổ đề.** Vùng `x` chứa vùng `y` ⇔ `x` thuộc đường tổ tiên `y, parent(y), parent(parent(y)), …, gốc`.
(⇐) mỗi bước lên cha là một quan hệ "chứa trực tiếp", ghép lại bằng tính bắc cầu; `x = y` dùng "chứa chính nó". (⇒) quan hệ chứa gián tiếp là một chuỗi chứa trực tiếp; vì mỗi vùng có ≤ 1 cha, chuỗi đó buộc phải là một đoạn của đường tổ tiên.

**Tính đúng của Cách 2.** `s ∪ {gốc}` là tập mọi vùng chứa `region1`. Khi đi từ `region2` lên, ta lần lượt gặp mọi vùng chứa `region2` theo thứ tự **kích thước tăng dần** (mỗi vùng chứa vùng trước). Vùng đầu tiên thuộc `s` (hoặc gốc, luôn chứa cả hai) là vùng chứa cả hai **sớm nhất** trên đường ⇒ nhỏ nhất. Mọi vùng chung khác nằm phía trên nó, tức chứa nó ⇒ lớn hơn hoặc bằng.

**Cách 3.** Sau khi cân bằng, `a` và `b` cùng độ sâu `d`. Các tổ tiên chung tạo thành đoạn từ LCA lên gốc; hai đường cùng độ dài, nên chúng chạm nhau lần đầu đúng ở độ sâu của LCA.

## 11. Phân tích Time / Space

Gọi `L = len(regions)` (≤ 10⁴), `N` = tổng số tên trong input (≤ 20·L = 2·10⁵), `h` = chiều cao cây (≤ N). Coi so sánh/băm một tên (≤ 20 ký tự) là `O(1)`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Brute force | `O(N + h²)` | `O(N)` |
| 2. Tập tổ tiên | `O(N)` | `O(N)` (bảng cha) + `O(h)` (set) |
| 3. Cân bằng độ sâu | `O(N)` | `O(N)` bảng cha + `O(1)` |
| 4. Hai con trỏ | `O(N)` | `O(N)` bảng cha + `O(1)` |
| 5. DFS từ gốc | `O(N)` | `O(N)` |

Phần chi phối là **dựng bảng cha** `O(N)`; các bước truy vấn chỉ `O(h)`. Nếu có `Q` truy vấn trên cùng cây: Cách 2 tốn `O(N + Q·h)`, binary lifting tốn `O(N log N + Q log N)`.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

Edge case quan trọng: một vùng chứa vùng kia (đáp án là chính nó), hai vùng anh em, một vùng là gốc, cây "dây chuyền".

In [17]:
ALL = {
    "brute": smallest_brute,
    "ancestor_set": smallest_ancestor_set,
    "depth": smallest_depth,
    "two_pointers": smallest_two_pointers,
    "dfs": smallest_dfs,
    "submit": lambda r, a, b: Solution().findSmallestRegion(r, a, b),
}
CHAIN = [["A", "B"], ["B", "C"], ["C", "D"], ["D", "E"]]
edge = [
    (REGIONS, "Canada", "Quebec", "Canada"),          # region1 chứa region2
    (REGIONS, "Quebec", "Canada", "Canada"),          # ngược lại
    (REGIONS, "Earth", "Brazil", "Earth"),            # một vùng là gốc
    (REGIONS, "Ontario", "Quebec", "Canada"),         # anh em
    (REGIONS, "Boston", "Brazil", "Earth"),
    (CHAIN, "E", "C", "C"),                           # dây chuyền
    ([["X", "Y", "Z"]], "Y", "Z", "X"),               # cây 1 tầng
    ([["B", "C"], ["A", "B", "D"]], "C", "D", "A"),   # danh sách cha xuất hiện sau con
]
for name, f in ALL.items():
    for regs, a, b, want in examples + edge:
        got = f(regs, a, b)
        assert got == want, (name, a, b, got, want)
print("Mọi cách đúng với", len(examples + edge), "test")

Mọi cách đúng với 10 test


### 12.2 Cây ngẫu nhiên — đối chiếu brute force (seed cố định)

Sinh cây ngẫu nhiên: nút `i` có cha là một nút `< i`; gom các con theo cha thành danh sách `[cha, con...]` (≤ 19 con mỗi danh sách, có thể tách thành nhiều danh sách cùng cha — đề không cấm), xáo trộn thứ tự danh sách.

In [18]:
def random_regions(n, rng, chain_bias=0.0):
    names = [f"R{i}" for i in range(n)]
    kids = defaultdict(list)
    for i in range(1, n):
        p = i - 1 if rng.random() < chain_bias else rng.randrange(i)
        kids[p].append(i)
    regions = []
    for p, cs in kids.items():
        for k in range(0, len(cs), 19):
            regions.append([names[p]] + [names[c] for c in cs[k:k + 19]])
    rng.shuffle(regions)
    return regions, names

rng = random.Random(1257)
for t in range(300):
    n = rng.randint(3, 60)
    regs, names = random_regions(n, rng, chain_bias=rng.choice([0, 0.5, 0.9]))
    a, b = rng.sample(names, 2)
    want = smallest_brute(regs, a, b)
    for name, f in ALL.items():
        assert f(regs, a, b) == want, (name, regs, a, b)
print("300 cây ngẫu nhiên: mọi cách khớp brute force")

300 cây ngẫu nhiên: mọi cách khớp brute force


### 12.3 Kích thước lớn: cây dây chuyền ~10⁴ tầng

Dây chuyền là trường hợp xấu cho DFS đệ quy (tràn stack — Cách 5 của ta dùng stack tường minh nên an toàn). Sau đó dùng cây hình chữ **Y** (hai nhánh dài, LCA ở gốc) — trường hợp xấu `O(h²)` của brute force.

In [19]:
n = 10001
chain = [[f"R{i}", f"R{i+1}"] for i in range(n - 1)]
a, b = f"R{n-1}", f"R{n-2}"
for name in ["ancestor_set", "depth", "two_pointers", "dfs", "submit"]:
    t0 = time.perf_counter()
    res = ALL[name](chain, a, b)
    print(f"{name:13} -> {res}  ({time.perf_counter() - t0:.3f}s)")
# Cây hình chữ Y: hai nhánh dài 1500 nối vào gốc -> brute force phải làm ~1500*1500 bước
Y = [["root", "L0", "M0"]] + [[f"L{i}", f"L{i+1}"] for i in range(1500)] + [[f"M{i}", f"M{i+1}"] for i in range(1500)]
for name in ["brute", "submit"]:
    t0 = time.perf_counter()
    res = ALL[name](Y, "L1500", "M1500")
    print(f"cây Y, {name:7} -> {res}  ({time.perf_counter() - t0:.3f}s)")

ancestor_set  -> R9999  (0.004s)
depth         -> R9999  (0.002s)
two_pointers  -> R9999  (0.002s)
dfs           -> R9999  (0.008s)
submit        -> R9999  (0.002s)


cây Y, brute   -> root  (0.153s)
cây Y, submit  -> root  (0.001s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Bỏ qua chính `region1` (quên "vùng chứa chính nó")

Bắt đầu tập tổ tiên từ `parent[region1]` thay vì `region1`: khi `region1` chứa `region2`, đáp án đúng là `region1` nhưng code trả về cha của nó.

In [20]:
def smallest_wrong1(regions, region1, region2):
    g = build_parent(regions)
    s = set()
    x = g.get(region1)                    # SAI: bỏ qua chính region1
    while x is not None:
        s.add(x)
        x = g.get(x)
    x = region2
    while x in g and x not in s:
        x = g[x]
    return x

print("Sai :", smallest_wrong1(REGIONS, "Canada", "Quebec"))
print("Đúng:", Solution().findSmallestRegion(REGIONS, "Canada", "Quebec"))

Sai : North America
Đúng: Canada


### Lỗi 2 — Dựng bảng cha sai chiều

Viết `g[r[0]] = y` trong vòng lặp con: mỗi cha chỉ nhớ **đứa con cuối cùng**, và quan hệ bị đảo ngược. Để không văng `KeyError`, ta dùng `.get` và giới hạn số bước — kết quả sai thấy rõ.

In [21]:
def smallest_wrong2(regions, region1, region2):
    g = {}
    for r in regions:
        for y in r[1:]:
            g[r[0]] = y                   # SAI: cha -> con
    s = set()
    x, steps = region1, 0
    while x in g and steps < 100:
        s.add(x); x = g[x]; steps += 1
    s.add(x)
    x, steps = region2, 0
    while x not in s and x in g and steps < 100:
        x = g[x]; steps += 1
    return x

print("Sai :", smallest_wrong2(REGIONS, "Quebec", "New York"))
print("Đúng:", Solution().findSmallestRegion(REGIONS, "Quebec", "New York"))

Sai : New York
Đúng: North America


### Lỗi 3 — Dùng `list` cho tập tổ tiên (đúng nhưng chậm) và đi hết đường thay vì dừng sớm

Một phiên bản khác hay gặp: lấy hai đường tổ tiên rồi tìm phần tử chung **cuối cùng** khi duyệt từ **lá lên** — sai thứ tự, trả về vùng lớn nhất (gốc) thay vì nhỏ nhất.

In [22]:
def smallest_wrong3(regions, region1, region2):
    parent = build_parent(regions)
    pa = path_to_root(region1, parent)
    pb = path_to_root(region2, parent)
    ans = None
    for x in pa:                          # từ lá lên gốc
        if x in pb:
            ans = x                       # SAI: ghi đè -> lấy phần tử chung CUỐI (gốc)
    return ans

print("Sai :", smallest_wrong3(REGIONS, "Quebec", "New York"))
print("Đúng:", Solution().findSmallestRegion(REGIONS, "Quebec", "New York"))

Sai : Earth
Đúng: North America


**Sửa:** dừng ở phần tử chung **đầu tiên** khi đi từ lá lên (`return x` ngay), hoặc duyệt từ gốc xuống và lấy phần tử chung cuối.

### Lỗi 4 — Cân bằng độ sâu nhưng quên bước "đi lên đồng thời" (hoặc chỉ so sánh cha)

Một số bạn nâng cho cùng độ sâu rồi trả về luôn cha của `a`, hoặc so sánh `parent[a] == parent[b]` — sai khi sau khi nâng `a == b` (một vùng chứa vùng kia).

In [23]:
def smallest_wrong4(regions, region1, region2):
    parent = build_parent(regions)
    def depth(x):
        d = 0
        while x in parent:
            x = parent[x]; d += 1
        return d
    a, b = region1, region2
    da, db = depth(a), depth(b)
    while da > db:
        a = parent[a]; da -= 1
    while db > da:
        b = parent[b]; db -= 1
    while parent.get(a) != parent.get(b):   # SAI: không kiểm tra a == b trước
        a, b = parent[a], parent[b]
    return parent.get(a, a)

print("Sai :", smallest_wrong4(REGIONS, "Canada", "Quebec"))
print("Đúng:", Solution().findSmallestRegion(REGIONS, "Canada", "Quebec"))

Sai : North America
Đúng: Canada


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Nhỏ nhất chứa cả hai", "quản lý chung gần nhất", "thư mục cha chung", "tổ tiên chung"** ⇒ LCA.
- Dữ liệu cho dạng **danh sách cha → con** và mỗi phần tử có ≤ 1 cha ⇒ dựng `parent` dict rồi **đi lên**; không cần dựng cây đầy đủ.
- Một truy vấn ⇒ **tập tổ tiên** (ngắn, ít lỗi). Bộ nhớ chặt ⇒ **cân bằng độ sâu** hoặc **hai con trỏ**.
- Nhiều truy vấn ⇒ **binary lifting** (`up[k][v]` = tổ tiên thứ `2^k`) hoặc Euler tour + RMQ.
- Luôn thử test "một vùng chứa vùng kia" — lỗi phổ biến nhất.

## 15. Biến thể của bài

1. **LeetCode 236 — LCA of a Binary Tree** (có con trỏ con, không có con trỏ cha) ⇒ DFS từ gốc.
2. **LeetCode 1650 — LCA of a Binary Tree III** (nút có con trỏ `parent`) ⇒ y hệt Cách 2/4.
3. **LeetCode 235 — LCA of a BST** ⇒ dùng thứ tự giá trị.
4. **Nhiều truy vấn LCA** ⇒ binary lifting (bài 1483 — Kth Ancestor of a Tree Node).
5. **Khoảng cách giữa hai vùng** = `depth(a) + depth(b) − 2·depth(LCA)`.

Minh hoạ biến thể 4 & 5 — binary lifting cho nhiều truy vấn:

In [24]:
class RegionLCA:
    def __init__(self, regions):
        parent = build_parent(regions)
        names = {r[0] for r in regions} | set(parent)
        self.id = {x: i for i, x in enumerate(sorted(names))}
        self.name = sorted(names)
        n = len(names)
        root = next(x for x in names if x not in parent)
        self.LOG = max(1, n.bit_length())
        up0 = [self.id[parent.get(x, root)] for x in self.name]
        self.up = [up0]
        for k in range(1, self.LOG):
            prev = self.up[-1]
            self.up.append([prev[prev[v]] for v in range(n)])
        self.depth = [0] * n
        children = defaultdict(list)
        for x, p in parent.items():
            children[p].append(x)
        q = deque([root])
        while q:
            x = q.popleft()
            for c in children[x]:
                self.depth[self.id[c]] = self.depth[self.id[x]] + 1
                q.append(c)

    def lca(self, a, b):
        u, v = self.id[a], self.id[b]
        if self.depth[u] < self.depth[v]:
            u, v = v, u
        diff = self.depth[u] - self.depth[v]
        for k in range(self.LOG):
            if diff >> k & 1:
                u = self.up[k][u]
        if u == v:
            return self.name[u]
        for k in reversed(range(self.LOG)):
            if self.up[k][u] != self.up[k][v]:
                u, v = self.up[k][u], self.up[k][v]
        return self.name[self.up[0][u]]

    def distance(self, a, b):
        c = self.lca(a, b)
        d = self.depth
        return d[self.id[a]] + d[self.id[b]] - 2 * d[self.id[c]]

T = RegionLCA(REGIONS)
print(T.lca("Quebec", "New York"), T.lca("Canada", "Quebec"), T.distance("Quebec", "Brazil"))
regs, names = random_regions(500, random.Random(5), chain_bias=0.7)
T = RegionLCA(regs)
r2 = random.Random(6)
for _ in range(500):
    a, b = r2.sample(names, 2)
    assert T.lca(a, b) == Solution().findSmallestRegion(regs, a, b)
print("Binary lifting khớp lời giải chuẩn trên 500 truy vấn")

North America Canada 5
Binary lifting khớp lời giải chuẩn trên 500 truy vấn


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space phụ | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Với mỗi tổ tiên của `region1`, đi từ `region2` lên kiểm tra | `O(N + h²)` | `O(N)` | Kiểm chứng, cây thấp |
| 2. Tập tổ tiên | Lưu tổ tiên `region1` vào set, đi từ `region2` lên tới khi gặp | `O(N)` | `O(N)` | Mặc định, ngắn gọn nhất |
| 3. Cân bằng độ sâu | Nâng vùng sâu hơn rồi đi lên đồng thời | `O(N)` | `O(1)` thêm | Khi tiết kiệm bộ nhớ, nền cho binary lifting |
| 4. Hai con trỏ | Đổi đường khi quá gốc (như bài 160) | `O(N)` | `O(1)` thêm | Code đẹp, không cần tính độ sâu |
| 5. DFS từ gốc | Đếm mục tiêu trong cây con, nút sâu nhất đủ 2 | `O(N)` | `O(N)` | Khi chỉ có con trỏ con (bài 236) |
| (+) Binary lifting | `up[k][v]` nhảy `2^k` | `O(N log N)` tiền xử lý, `O(log N)`/truy vấn | `O(N log N)` | Nhiều truy vấn |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
Với `REGIONS` của đề, `findSmallestRegion(REGIONS, "Ontario", "Boston")` trả về gì?

### Bài 2 (tính tay)
Cho `regs = [["W","X","Y"],["X","P","Q"],["Q","M"],["Y","Z"]]`. Tính tay đáp án cho (`M`, `P`), (`M`, `Z`), (`Q`, `M`). Viết đường tổ tiên của từng vùng.

### Bài 3 (điền TODO)
Hoàn thiện `region_distance(regions, a, b)`: số bước đi (lên/xuống) ngắn nhất giữa hai vùng trên cây.

### Bài 4 (tự viết lại)
Viết lại Cách 2 nhưng **đi từ `region2` vào set trước**, rồi đi từ `region1`; đặt tên `smallest_swapped`. Kết quả có đổi không?

### Bài 5 (mini challenge)
Viết `smallest_common_many(regions, targets)`: vùng nhỏ nhất chứa **mọi** vùng trong danh sách `targets` (≥ 1 phần tử).

In [25]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def region_distance(regions, a, b):
    parent = build_parent(regions)
    # TODO 1: tìm LCA c của a và b
    # TODO 2: đếm số bước từ a lên c và từ b lên c, trả về tổng
    pass

# Bài 4 — TODO
def smallest_swapped(regions, region1, region2):
    # TODO
    pass

# Bài 5 — TODO
def smallest_common_many(regions, targets):
    # TODO: gộp dần: ans = targets[0]; ans = LCA(ans, t) cho các t còn lại
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Đường của Ontario: Ontario → Canada → North America → Earth. (2) Đường của Boston: Boston → United States → North America → … (3) Chung đầu tiên: North America.

**Bài 2.** (1) M → Q → X → W; P → X → W; Z → Y → W. (2) (M, P): chung đầu tiên khi đi từ P là X. (3) (M, Z) → W; (Q, M) → Q (Q chứa M).

**Bài 3.** (1) Dùng `Solution().findSmallestRegion`. (2) Viết hàm đếm bước `steps(x, c)` bằng vòng `while x != c`. (3) Tổng hai số bước.

**Bài 4.** (1) Chỉ cần hoán đổi tham số. (2) LCA đối xứng. (3) Kết quả không đổi — assert trên test ngẫu nhiên.

**Bài 5.** (1) LCA có tính kết hợp: `LCA(a, b, c) = LCA(LCA(a, b), c)`. (2) Cẩn thận khi chỉ có 1 phần tử: đáp án là chính nó. (3) Để nhanh hơn với nhiều target: dựng bảng cha một lần, tái sử dụng.

## 19. Đáp án tham khảo

In [26]:
# Bài 1
assert Solution().findSmallestRegion(REGIONS, "Ontario", "Boston") == "North America"
# Bài 2
B2 = [["W", "X", "Y"], ["X", "P", "Q"], ["Q", "M"], ["Y", "Z"]]
assert Solution().findSmallestRegion(B2, "M", "P") == "X"
assert Solution().findSmallestRegion(B2, "M", "Z") == "W"
assert Solution().findSmallestRegion(B2, "Q", "M") == "Q"

def lca_with_parent(parent, a, b):
    s = set()
    x = a
    while True:
        s.add(x)
        if x not in parent:
            break
        x = parent[x]
    x = b
    while x not in s:
        x = parent[x]
    return x

# Bài 3
def region_distance_sol(regions, a, b):
    parent = build_parent(regions)
    c = lca_with_parent(parent, a, b)
    def steps(x):
        k = 0
        while x != c:
            x = parent[x]; k += 1
        return k
    return steps(a) + steps(b)

# Bài 4
def smallest_swapped_sol(regions, region1, region2):
    return Solution().findSmallestRegion(regions, region2, region1)

# Bài 5
def smallest_common_many_sol(regions, targets):
    parent = build_parent(regions)
    ans = targets[0]
    for t in targets[1:]:
        ans = lca_with_parent(parent, ans, t)
    return ans

assert region_distance_sol(REGIONS, "Quebec", "Brazil") == 5
assert region_distance_sol(REGIONS, "Canada", "Quebec") == 1
assert smallest_common_many_sol(REGIONS, ["Quebec", "Boston", "Ontario"]) == "North America"
assert smallest_common_many_sol(REGIONS, ["Brazil"]) == "Brazil"
rng = random.Random(4242)
for _ in range(200):
    regs, names = random_regions(rng.randint(3, 40), rng, 0.3)
    a, b = rng.sample(names, 2)
    assert smallest_swapped_sol(regs, a, b) == Solution().findSmallestRegion(regs, a, b)
    T = RegionLCA(regs)
    assert region_distance_sol(regs, a, b) == T.distance(a, b)
    ts = rng.sample(names, rng.randint(1, min(4, len(names))))
    want = ts[0]
    for t in ts[1:]:
        want = smallest_brute(regs, want, t)
    assert smallest_common_many_sol(regs, ts) == want
print("Đáp án Bài 1–5: mọi assert đều qua")

Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template LCA khi có bảng cha:**

```python
parent = {}
for r in regions:
    for y in r[1:]:
        parent[y] = r[0]

anc = set()
x = a
while x in parent:      # gốc không cần thêm: vòng sau tự dừng ở gốc
    anc.add(x)
    x = parent[x]
x = b
while x in parent and x not in anc:
    x = parent[x]
return x
```

**Template cân bằng độ sâu:** tính `depth`, nâng vùng sâu hơn, rồi `while a != b: a, b = parent[a], parent[b]`.

**Checklist trước khi submit:**
- [ ] Bảng cha đúng chiều: `parent[con] = cha` (`cha = r[0]`).
- [ ] Tập tổ tiên **bắt đầu từ chính `region1`** (vùng chứa chính nó).
- [ ] Điều kiện dừng ở gốc (gốc không có trong `parent`) — không `KeyError`.
- [ ] Test "một vùng chứa vùng kia" và "hai vùng anh em".
- [ ] Danh sách của cha có thể xuất hiện **sau** danh sách của con — không giả định thứ tự.
- [ ] Đúng chữ ký: `findSmallestRegion(self, regions, region1, region2) -> str`.